In [36]:
## Baseline Data Analysis

## This section analyzes labor market baseline indicators by period and economic activity.  
## The main metrics are employee count, open job count, open job rate and open job distribution.

In [37]:
import pandas as pd

In [38]:
file_path = r"C:/Users/emree/sql-basic-practice/sgk_labor_market_excel_dashboard.xlsm"

In [39]:
file_path

'C:/Users/emree/sql-basic-practice/sgk_labor_market_excel_dashboard.xlsm'

In [40]:
excel_file = pd.ExcelFile(file_path)

In [41]:
excel_file

In [42]:
sheet_names = excel_file.sheet_names

In [43]:
sheet_names

['25-26-serviceproductionbysec',
 '25-26-employmentbysector',
 '25-26-quarterlybaselinedata',
 '2025-2026-skill',
 'pivotparametre',
 'skillparametre',
 'dashboard']

In [44]:
working_sheets = [
    "25-26-serviceproductionbysec",
    "25-26-employmentbysector",
    "25-26-quarterlybaselinedata",
    "2025-2026-skill"
]
working_sheets

['25-26-serviceproductionbysec',
 '25-26-employmentbysector',
 '25-26-quarterlybaselinedata',
 '2025-2026-skill']

In [45]:
# Load the baseline data sheet
baseline_sheet = "25-26-quarterlybaselinedata"
df_baseline = pd.read_excel(file_path, sheet_name = baseline_sheet)
df_baseline.head()

,Dönem,Ekonomik Faaliyet,Çalışan Sayısı,Açık İş Sayısı,Açık İş Oranı,Açık İş Dağılımı,Tbl_Anahtar
0,2026Q1,İmalat,3589558,81303,0.022,0.300,2026Q1|İmalat
1,2026Q1,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,4088180,56402,0.014,0.208,2026Q1|Toptan ve Perakende Ticaret; Motorlu Ka...
2,2026Q1,Konaklama ve Yiyecek Hizmeti Faaliyetleri,3311283,53911,0.016,0.199,2026Q1|Konaklama ve Yiyecek Hizmeti Faaliyetleri
3,2026Q1,"Mesleki, Bilimsel ve Teknik Faaliyetler",1107366,17351,0.015,0.064,"2026Q1|Mesleki, Bilimsel ve Teknik Faaliyetler"
4,2026Q1,İnşaat,1618013,14175,0.009,0.052,2026Q1|İnşaat


In [46]:
df_baseline.shape

(86, 7)

In [47]:
df_baseline = df_baseline.rename(columns={
    "Dönem" : "period",
    "Ekonomik Faaliyet" : "economic_activity",
    "Çalışan Sayısı" : "employee_count",
    "Açık İş Sayısı" : "open_job_count",
    "Açık İş Oranı" : "open_job_rate",
    "Açık İş Dağılımı" : "open_job_distribution",
    "Tbl_Anahtar" : "table_key"
})
df_baseline.head()

,period,economic_activity,employee_count,open_job_count,open_job_rate,open_job_distribution,table_key
0,2026Q1,İmalat,3589558,81303,0.022,0.300,2026Q1|İmalat
1,2026Q1,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,4088180,56402,0.014,0.208,2026Q1|Toptan ve Perakende Ticaret; Motorlu Ka...
2,2026Q1,Konaklama ve Yiyecek Hizmeti Faaliyetleri,3311283,53911,0.016,0.199,2026Q1|Konaklama ve Yiyecek Hizmeti Faaliyetleri
3,2026Q1,"Mesleki, Bilimsel ve Teknik Faaliyetler",1107366,17351,0.015,0.064,"2026Q1|Mesleki, Bilimsel ve Teknik Faaliyetler"
4,2026Q1,İnşaat,1618013,14175,0.009,0.052,2026Q1|İnşaat


In [48]:
df_baseline.columns

Index(['period', 'economic_activity', 'employee_count', 'open_job_count',
       'open_job_rate', 'open_job_distribution', 'table_key'],
      dtype='object')

## Data Quality Check

Before analysis, missing values and duplicate rows are checked to understand the reliability of the dataset.

In [49]:
df_baseline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   period                 86 non-null     object 
 1   economic_activity      86 non-null     object 
 2   employee_count         86 non-null     int64  
 3   open_job_count         86 non-null     int64  
 4   open_job_rate          86 non-null     float64
 5   open_job_distribution  86 non-null     float64
 6   table_key              86 non-null     object 
dtypes: float64(2), int64(2), object(3)
memory usage: 4.8+ KB


In [50]:
df_baseline.isnull().sum()

period                   0
economic_activity        0
employee_count           0
open_job_count           0
open_job_rate            0
open_job_distribution    0
table_key                0
dtype: int64

In [51]:
df_baseline.duplicated().sum()

np.int64(0)

### Analysis 1: Period-Level Summary

This analysis summarizes total employee count and total open job count by period.  
It helps understand the overall labor market situation for each period.

In [52]:
period_summary = df_baseline.groupby("period").agg(
    total_employee_count=("employee_count", "sum"),
    total_open_job_count=("open_job_count", "sum"),
    average_open_job_rate=("open_job_rate", "mean"),
    total_open_job_distribution=("open_job_distribution", "sum")
).reset_index()

period_summary

,period,total_employee_count,total_open_job_count,average_open_job_rate,total_open_job_distribution
0,2025Q1,18960678,343769,0.013941,0.998
1,2025Q2,18345437,206189,0.009353,1.000
2,2025Q3,18745331,200184,0.008706,1.001
3,2025Q4,18157176,130426,0.005824,1.001
4,2026Q1,19274912,271291,0.011056,0.998


### Analysis 2: Sectors with the Highest Open Job Count

This analysis identifies which economic activities have the highest number of open jobs.  
It shows where labor demand is strongest in absolute numbers.

In [53]:
top_open_jobs = df_baseline[
    ["period","economic_activity","employee_count","open_job_count","open_job_rate","open_job_distribution"]
].sort_values(by = "open_job_count", ascending=False)

top_open_jobs.head(10)

,period,economic_activity,employee_count,open_job_count,open_job_rate,open_job_distribution
69,2025Q1,İmalat,4505367,118588,0.026,0.345
0,2026Q1,İmalat,3589558,81303,0.022,0.300
70,2025Q1,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,3597552,66784,0.018,0.194
1,2026Q1,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,4088180,56402,0.014,0.208
2,2026Q1,Konaklama ve Yiyecek Hizmeti Faaliyetleri,3311283,53911,0.016,0.199
35,2025Q3,İmalat,4564239,50964,0.011,0.255
52,2025Q2,İmalat,4582077,48868,0.011,0.237
71,2025Q1,İnşaat,2117573,41996,0.019,0.122
53,2025Q2,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,3452617,37305,0.011,0.181
54,2025Q2,Konaklama ve Yiyecek Hizmeti Faaliyetleri,1374270,33268,0.024,0.161


The manufacturing sector appears as one of the strongest sectors in terms of open job count, indicating high labor demand.

### Analysis 3: Sectors with the Highest Open Job Rate

This analysis focuses on open job rate instead of total open job count.  
It helps compare labor demand relative to employment size.

In [54]:
top_open_job_rate = df_baseline [
    ["period","economic_activity", "employee_count","open_job_count","open_job_rate","open_job_distribution"]
].sort_values(by = "open_job_rate", ascending=False)

top_open_job_rate.head(10)

,period,economic_activity,employee_count,open_job_count,open_job_rate,open_job_distribution
5,2026Q1,Diğer Hizmet Faaliyetleri,367521,10111,0.027,0.037
69,2025Q1,İmalat,4505367,118588,0.026,0.345
73,2025Q1,Konaklama ve Yiyecek Hizmeti Faaliyetleri,1170921,28873,0.024,0.084
54,2025Q2,Konaklama ve Yiyecek Hizmeti Faaliyetleri,1374270,33268,0.024,0.161
0,2026Q1,İmalat,3589558,81303,0.022,0.300
71,2025Q1,İnşaat,2117573,41996,0.019,0.122
76,2025Q1,Diğer Hizmet Faaliyetleri,369770,7253,0.019,0.021
70,2025Q1,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,3597552,66784,0.018,0.194
41,2025Q3,"Mesleki, Bilimsel ve Teknik Faaliyetler",867127,15899,0.018,0.079
82,2025Q1,"Kültür, Sanat Eğlence, Dinlence ve Spor",94994,1559,0.016,0.005


### Analysis 4: Sectors with the Highest Employee Count

This analysis identifies the largest sectors by employee count.  
It provides context for interpreting open job demand.

In [56]:
top_employee_count = df_baseline [
    ["period","economic_activity","employee_count","open_job_count","open_job_rate","open_job_distribution"]
].sort_values(by = "employee_count", ascending=False)

top_employee_count.head(10)

,period,economic_activity,employee_count,open_job_count,open_job_rate,open_job_distribution
52,2025Q2,İmalat,4582077,48868,0.011,0.237
35,2025Q3,İmalat,4564239,50964,0.011,0.255
69,2025Q1,İmalat,4505367,118588,0.026,0.345
18,2025Q4,İmalat,4477895,31494,0.007,0.241
1,2026Q1,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,4088180,56402,0.014,0.208
70,2025Q1,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,3597552,66784,0.018,0.194
0,2026Q1,İmalat,3589558,81303,0.022,0.300
53,2025Q2,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,3452617,37305,0.011,0.181
36,2025Q3,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,3450667,32406,0.009,0.162
20,2025Q4,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,3352480,22142,0.007,0.170


### Analysis 5: Open Jobs per 1,000 Employees

This analysis calculates open jobs per 1,000 employees to compare labor demand across sectors more fairly.

In [59]:
df_baseline["open_jobs_per_1000_employees"] = (
    df_baseline["open_job_count"] / df_baseline["employee_count"] * 1000
)

open_jobs_per_1000 = df_baseline[
    ["period","economic_activity","employee_count","open_job_count","open_jobs_per_1000_employees"]
].sort_values(by = "open_jobs_per_1000_employees", ascending=False)

open_jobs_per_1000.head(10)

,period,economic_activity,employee_count,open_job_count,open_jobs_per_1000_employees
5,2026Q1,Diğer Hizmet Faaliyetleri,367521,10111,27.511353
69,2025Q1,İmalat,4505367,118588,26.321496
73,2025Q1,Konaklama ve Yiyecek Hizmeti Faaliyetleri,1170921,28873,24.658367
54,2025Q2,Konaklama ve Yiyecek Hizmeti Faaliyetleri,1374270,33268,24.207761
0,2026Q1,İmalat,3589558,81303,22.649864
71,2025Q1,İnşaat,2117573,41996,19.832138
76,2025Q1,Diğer Hizmet Faaliyetleri,369770,7253,19.614896
70,2025Q1,Toptan ve Perakende Ticaret; Motorlu Kara Taşı...,3597552,66784,18.563734
41,2025Q3,"Mesleki, Bilimsel ve Teknik Faaliyetler",867127,15899,18.335261
80,2025Q1,Gayrimenkul Faaliyetleri,166967,2779,16.644007


## Key Insights

- Total employee count increased from 18.96 million in 2025 Q1 to 19.27 million in 2026 Q1.
- Total open job count was highest in 2025 Q1 with 343,769 open jobs.
- Manufacturing appears as one of the strongest sectors in terms of both employee count and open job count.
- Other Service Activities and Manufacturing show high open jobs per 1,000 employees, indicating stronger relative labor demand.
- Open job rate analysis provides a fairer comparison between sectors by considering labor demand relative to employment size.